In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from src.utils import show_confusion_matrix,plot_roc_curve
from sklearn.preprocessing import StandardScaler
import copy
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

In [ ]:
raw_path = os.path.dirname(os.path.abspath('.'))

probas_save = "..\probas"
preds_save = "..\preds"

df_FE_train_path = os.path.join(raw_path, "df_FE_train.csv")
df_FE_test_path = os.path.join(raw_path, 'df_FE_test.csv')
df_FE = pd.read_csv(df_FE_train_path)
df_FE_test = pd.read_csv(df_FE_test_path)
df_FE_test.drop(['Stress_Level'], axis=1, inplace=True)

X = df_FE.drop(['Stress_Level'], axis=1)
y = df_FE['Stress_Level']

target_names = ['0', '1']
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X,y)

In [ ]:
CONFIG = {
    "batch_size" : 512,
    "epochs": 120,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "patience": 7,
    "n_splits": 10,
    "random_state": 42,
    "num_workers": 0, # Num workers are not working on notebooks, please increase it when you need to run the NN Train fucntion in py
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
class TabularDataset(Dataset):
    """
    PyTorch Dataset for tabular feature arrays.
    
    Args:
        X: feature array of shape (n_samples, n_features)
        y: optional label array of shape (n_samples,); omit for inference-only datasets
    """
    def __init__(self, X: np.ndarray, y:np.ndarray=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [ ]:
class Net(nn.Module):
    """
    Residual MLP classifier for tabular data.
    
    Args:
        input_dim: number of input features
        num_classes: number of output classes
        hidden_dims: width of each block in the network
        dropout: dropout probability used througout
    """
    def __init__(self, input_dim:int, num_classes:int=2, hidden_dims:list=[64,128,256,128,64,], dropout:float=0.2):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0], bias=False),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.blocks = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            in_d = hidden_dims[i]
            out_d = hidden_dims[i+1]
            block = nn.Sequential(
                nn.Linear(in_d, out_d, bias=False),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(out_d, out_d, bias=False),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            shortcut = (
                nn.Linear(in_d,out_d,bias=False)
                if in_d != out_d else nn.Identity()
            )
            self.blocks.append(nn.ModuleDict({
                "block": block,
                "shortcut": shortcut
            }))

        self.head = nn.Linear(hidden_dims[-1], num_classes)

    def forward(self,x):
        x = self.input_proj(x)
        for m in self.blocks:
            residual = m["shortcut"](x)
            x = m["block"](x) + residual
        return self.head(x)

In [ ]:
def train_epoch(model: nn.Module, loader, criterion, optimizer, scaler):
    """
    Run one training epoch with mixed precision and gradient clipping.

    Args:
        model: the network to train
        loader: training DataLoader
        criterion: loss function
        optimizer: optimizer
        scaler: torch.amp.GradScaler for mixed precision

    Returns:
        (avg_loss, accuracy) for the epoch
    """
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total+= len(y_batch)

    return total_loss / total, total_correct / total


In [ ]:
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    """
    Run one eval pass (no gradient updates).
    
    Args:
        model: the network to evaluate
        loader: validation/test Dataloader
        criterion: loss function
        
    Returns:
        (avg_loss, accuracy, preds, probas)
    """
    model.eval()
    total_loss, total_correct, total = 0.0,0,0

    all_preds = []
    all_probas = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type='cuda'):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

        probas = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)
        all_preds.extend(preds.cpu().numpy())
        all_probas.append(probas.cpu().numpy())

    all_probas = np.concatenate(all_probas, axis=0)

    return total_loss / total, total_correct / total, np.array(all_preds), all_probas

In [ ]:
def train_pipeline_nn(X:pd.DataFrame, y:pd.Series, config:dict=CONFIG):
    """Train a Net with stratified cross-validation and early stopping.
    
    Args:
        X: feature Dataset
        y: target Series
        config: training config dict
        
    Returns:
        (oof_predsl, oof_probas, all_history, mean_score)
    """

    X_arr = X.values.astype(np.float32)
    y_arr = y.values.astype(np.int64)

    input_dim = X_arr.shape[1]
    num_classes = len(np.unique(y_arr))

    cv = StratifiedKFold(
        n_splits = config['n_splits'],
        shuffle = True,
        random_state=config['random_state'],
    )

    oof_probas = np.zeros((len(y_arr), num_classes))
    oof_preds = np.zeros(len(y_arr), dtype=np.int64)
    all_scores = []
    all_history = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_arr, y_arr), start=1):
        print(f"\n{'='*40}")
        print(f"Fold {fold} out of {config['n_splits']}")
        print(f"{'='*40}")

        X_tr, X_vl = X_arr[train_idx], X_arr[valid_idx]
        y_tr, y_vl = y_arr[train_idx], y_arr[valid_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_vl = scaler.transform(X_vl)

        #DataLoaders
        train_ds = TabularDataset(X_tr, y_tr)
        valid_ds = TabularDataset(X_vl, y_vl)

        train_loader = DataLoader(
            train_ds,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=config['num_workers'],
            pin_memory=True,
        )
        valid_loader = DataLoader(
            valid_ds,
            batch_size=config["batch_size"],
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=True,)

        #Model
        model = Net(input_dim=input_dim, num_classes=num_classes).to(DEVICE)

        criterion = nn.CrossEntropyLoss()

        optimiser = optim.AdamW(
            model.parameters(),
            lr=config['lr'],
            weight_decay=config['weight_decay']
        )

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimiser,
            T_max=config['epochs'],
            eta_min=1e-6,
        )

        amp_scaler = torch.amp.GradScaler()

        best_val_loss = float('inf')
        best_val_acc = 0.0
        best_weights = None
        patience_count = 0
        history = {"train_loss": [], "train_acc": [],
                    "val_loss": [],   "val_acc": []}

        for epoch in range(1, config['epochs']+1):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimiser, amp_scaler)
            vl_loss, vl_acc, _, _ = eval_epoch(model, valid_loader, criterion)
            scheduler.step()

            history['train_loss'].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(vl_loss)
            history["val_acc"].append(vl_acc)

            print(f"Epoch {epoch:>3} out of {config['epochs']} | Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")

            #Early stopping on val_loss
            if vl_loss < best_val_loss:
                best_val_loss = vl_loss
                best_val_acc = vl_acc
                best_weights = copy.deepcopy(model.state_dict())
                patience_count = 0
            else:
                patience_count += 1
                if patience_count >= config['patience']:
                    print(f"\nEarly stopping at epoch {epoch}")
                    break

        model.load_state_dict(best_weights)
        _, _, fold_preds, fold_probas = eval_epoch(model, valid_loader, criterion)
        oof_preds[valid_idx] = fold_preds
        oof_probas[valid_idx] = fold_probas

        fold_score = balanced_accuracy_score(y_vl, fold_preds)
        all_scores.append(fold_score)
        all_history.append(history)

        print(f"\nFold {fold} Best Val Acc: {best_val_acc:.5f}")
        print(f"Fold {fold} Final Score: {fold_score:.5f}")

    mean_score = np.mean(all_scores)
    print(f"\n{'='*40}")
    print(f"Mean CV Accuracy: {mean_score:.5f}")
    print(f"{'='*40}\n")

    return oof_preds, oof_probas, all_history, mean_score

In [ ]:
oof_nn_preds, oof_nn_probas, history_nn, cv_score_nn = train_pipeline_nn(X_resampled, y_resampled)

In [ ]:
def train_final_nn(X:pd.DataFrame, y:pd.Series, config:dict):
    """
    Train Net on the full dataset (no validation split) for fnal test predictions.
    
    Args:
        X: feature DataFrame
        y: target Series
        config: training config dict
        
    Returns:
        (trained model, fitted StandartScaler)
    """

    X_arr = X.values.astype(np.float32)
    y_arr = y.values.astype(np.int64)

    scaler = StandardScaler()
    X_arr = scaler.fit_transform(X_arr)

    dataset = TabularDataset(X_arr, y_arr)
    loader = DataLoader(dataset, batch_size=config['batch_size']*2,
                        shuffle=True, num_workers=config['num_workers'],
                        pin_memory=True)

    model = Net(input_dim=X_arr.shape[1], num_classes=2).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimiser = optim.AdamW(model.parameters(), lr=config['lr'],
                            weight_decay=config['weight_decay'])

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimiser,
        T_max=config['epochs'],
        eta_min=1e-6
    )

    amp_scaler = torch.amp.GradScaler()

    for epoch in range(1, config['epochs'] + 1):
        train_epoch(model, loader, criterion, optimiser, amp_scaler)
        scheduler.step()

    return model, scaler

In [ ]:
final_nn_model, final_scaler = train_final_nn(X_resampled, y_resampled,CONFIG)

In [ ]:
X_test_arr = final_scaler.transform(df_FE_test.values.astype(np.float32))
test_dataset = TabularDataset(X_test_arr)
test_loader  = DataLoader(test_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=False)

final_nn_model.eval()
nn_test_probas = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits  = final_nn_model(X_batch)
        probas  = torch.softmax(logits, dim=1)
        nn_test_probas.append(probas.cpu().numpy())

In [ ]:
np.save(os.path.join(preds_save, "oof_nn_preds.npy"), oof_nn_preds)
np.save(os.path.join(probas_save, "oof_nn_probas.npy"), oof_nn_probas)
# concatenate list of batch arrays into a single array before saving
nn_test_probas = np.concatenate(nn_test_probas, axis=0)
np.save(os.path.join(probas_save, "nn_test_probas.npy"), nn_test_probas)

In [ ]:
h = history_nn[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(h['train_loss'], label='Train'); axes[0].plot(h['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(h['train_acc'], label='Train'); axes[1].plot(h['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.show()

show_confusion_matrix(y_resampled, oof_nn_preds, target_names)
plot_roc_curve(y_resampled, oof_nn_probas, target_names)
print(classification_report(y_resampled, oof_nn_preds))